# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavithamuddagowni3-cyber/flyrank/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
## 1. Method choice and why

### Selected method: Random Forest Classification

This project uses a Random Forest classifier because the lane is a **content refresh opportunity scoring task**, where the goal is to rank pages that are more likely to show decline signals.

Random Forest fits this problem because:

- It can capture complex relationships between multiple content signals such as impressions, CTR, age, position, sessions, and engagement.
- It handles non-linear patterns better than simple if-else rules.
- It works well with mixed feature types after preprocessing.
- It provides feature importance, helping explain which signals influence predictions.

The model predicts:

`is_declining_label`

where:

- `1` = page shows declining trend based on the defined label.
- `0` = page does not show declining trend.

The Random Forest model is compared against the transparent baseline rule to check whether machine learning provides better ranking performance.

This is a decision-support system, not an automatic content update system.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
## 2. Split design

### Selected split: Client-group holdout split

The model evaluation uses a client-level grouped split.

Instead of randomly splitting individual pages, all pages from the same client are kept together in either training or testing data.

### Why this split is honest

Pages from the same client can share patterns such as:
- similar content strategies,
- website characteristics,
- search behavior,
- traffic patterns.

If pages from the same client appear in both training and testing sets, the model may memorize client-specific patterns and produce overly optimistic results.

By holding out complete clients, the evaluation tests whether the model can generalize to content from clients it has not seen before.

### Limitation

This split is suitable for testing cross-client generalization. For a future prediction task (for example, predicting next month's decline), a time-aware split would also be important to ensure the model only learns from past data.

# Check split strategy from model results

import json
from pathlib import Path

results_path = Path("../../outputs/model_results.json")

with open(results_path, "r") as f:
    results = json.load(f)

print("Split strategy:")
print(results.get("split_strategy", "Not found"))

print("\nModel evaluation:")
print(results)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
## 3. Train + compare vs my baseline

The machine learning models were evaluated against the Week-4 baseline using the same dataset, client-holdout split, and ranking metrics.

The baseline represents a transparent hand-written scoring rule, while the ML models learn patterns from multiple observable signals.

### Model comparison

| Method | ROC AUC | Average Precision | Precision@50 |
|---|---:|---:|---:|
| Baseline rules | 0.627 | 0.468 | 0.240 |
| Logistic Regression | 0.700 | 0.522 | 0.400 |
| Decision Tree | 0.742 | 0.575 | 0.540 |
| Random Forest | 0.750 | 0.618 | 0.740 |

### Interpretation

The Random Forest model performs best among the tested approaches and improves Precision@50 compared with the baseline rule.

This means that when reviewing the top 50 ranked pages, the ML model identifies more pages matching the decline label than the fixed rule.

The improvement suggests that combining multiple signals can create a stronger prioritization system than a single manually designed scoring formula.

However, the model remains a decision-support tool. Better ranking performance does not prove that a content refresh will cause recovery.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
## 4. Errors and interpretation

The model is not always correct because it learns patterns from historical signals, not the true reasons behind content performance changes.

### Where the model can be wrong

- **False positives:** Some pages may be ranked as decline risks but are actually healthy. For example, seasonal content or pages affected by temporary search demand changes may look risky.
- **False negatives:** Some pages may not be flagged even though they later decline because important factors were not available in the dataset.
- **Low-volume pages:** Pages with limited impressions or sessions can show unstable patterns because small changes create large percentage differences.

### What the model relies on

The Random Forest model mainly uses observable signals such as:
- search visibility (impressions and clicks);
- CTR and average position;
- content age and freshness;
- sessions and engagement metrics;
- content structure features.

These signals help identify patterns associated with decline, but they do not explain causation.

### Practical interpretation

The model should be used to create a review queue for content teams. A high score means "investigate this page first," not "this page will definitely recover after a refresh."

import pandas as pd

# Load predictions
pred = pd.read_csv("../../data/processed/model_predictions.csv")

# Show false positives and false negatives
false_positives = pred[
    (pred["prediction"] == 1) &
    (pred["is_declining_label"] == 0)
]

false_negatives = pred[
    (pred["prediction"] == 0) &
    (pred["is_declining_label"] == 1)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nExample false positives:")
display(false_positives.head())

print("\nExample false negatives:")
display(false_negatives.head())

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.